# Grad-CAM Validation & ROI Agreement

**Production pipeline this notebook implements and tests:**

```
full mammogram (ANY size) -> preprocess -> model -> Grad-CAM -> heatmap at the ORIGINAL size
```

The model takes **only the full image**. The cropped lesion patch and the ROI mask are
training-time signals; neither is needed here. The ROI mask appears in this notebook purely as
*ground truth to score the heatmap against* -- it is never fed to the model.

### The two questions this notebook answers

1. **Does the heatmap come back at the right size?** The model sees a 384x640 padded tensor, but a
   user hands us e.g. a 3024x5063 mammogram. The explanation must be overlayable on *their* image,
   so the padding has to be stripped and the map resized back. `cam_to_original` inverts the
   preprocessing geometry exactly.
2. **Is the heatmap actually pointing at the lesion?** Measured, not eyeballed -- pointing game,
   energy concentration, concentration-ratio-vs-chance, IoU and pixel AP against the ROI masks,
   with paired significance tests when two models are compared.

### Why measurement matters

Nearly every CBIS-DDSM paper that shows Grad-CAM shows three hand-picked heatmaps and asserts they
look right. A quantified, statistically tested localization result is a stronger contribution than
a couple of points of accuracy -- and unlike accuracy, it is largely within our control.

**What can run today, and how NOT to read it.** The hi-res single-input models are not trained yet,
so everything here runs against the surviving dual-input teacher's `full_extractor`, which has the
identical interface to `SingleInputModel.extractor`.

This exercises every line of the pipeline on real data, which is its purpose. But the resulting
localization numbers are **a code check, not a scientific baseline**, because the teacher is being
used far outside its training distribution in two independent ways:

1. **Wrong input scale.** It was trained at 224x224, where a lesion spans ~19x10 px. Here it is fed
   640x384, where the same lesion spans ~35x34 px. Convolutions accept any input size, so this runs
   without error -- but every learned feature is tuned to the wrong scale.
2. **Zeroed crop branch.** The teacher always saw a real lesion crop in its second input during
   training; here that branch is fed zeros. It never encountered this at training time.

So expect the teacher to score only slightly above a randomly-initialized model. That is the
*correct* outcome and does not indicate the harness is broken -- the harness is validated by the
checks in section 3 (random scores at chance, CAM peaks track a known input patch). Do not quote
these teacher figures in the thesis. The real numbers come from the hi-res single-input models;
when those checkpoints exist, only the `load_model` cell changes.

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath("../common"))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import hires_lib as H

warnings.filterwarnings("ignore", category=FutureWarning)
Image.MAX_IMAGE_PIXELS = None

ROOT = os.path.abspath("../..")
PNG_ROOT = os.path.join(ROOT, "cbis_ddsm_png")
CKPT_DIR = os.path.join(ROOT, "notebooks", "experiment_1_baseline", "checkpoints")
RESULTS_DIR, FIG_DIR = "results", "figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

H.set_seed()
print("torch", torch.__version__, "| device", H.DEVICE)
print(f"cache {H.HIRES_W} x {H.HIRES_H} (W x H)")

In [ ]:
test_df = pd.read_csv(os.path.join(ROOT, "dataframes", "test_df_hires.csv"))
val_df = pd.read_csv(os.path.join(ROOT, "dataframes", "val_df_hires.csv"))
print("test:", len(test_df), "| val:", len(val_df))

_BS = chr(92)


def winlong(p):
    """CBIS-DDSM's nested DICOM UID folders blow past Windows MAX_PATH."""
    p = os.path.abspath(p)
    return (_BS * 2 + "?" + _BS + p) if not p.startswith(_BS * 2) else p


def source_path(row):
    return os.path.join(PNG_ROOT, str(row["image file path"]).replace("/", os.sep))


_size_cache = {}


def source_size(row):
    """(W, H) of the ORIGINAL png. Cached -- reopening a 15MP png per call is slow."""
    p = source_path(row)
    if p not in _size_cache:
        with Image.open(winlong(p)) as im:
            _size_cache[p] = im.size
    return _size_cache[p]


print("example source size (W x H):", source_size(test_df.iloc[0]))

## 1. The production path

`explain(image_path)` is the deployable function: raw image in, probability + original-resolution
heatmap out. Everything else exists to test it.

`cam_to_original` inverts `fit_pad`. The geometry is fully determined by the source dimensions, so
no state has to be carried between preprocessing and postprocessing.

In [ ]:
def preprocess(pil_gray):
    """Raw PIL image -> (1,3,640,384) normalized tensor, identical geometry to the cache builder,
    so a model trained on the cache sees exactly this at inference."""
    canvas = H.fit_pad(pil_gray, Image.LANCZOS)
    t = H.TF.to_tensor(canvas)
    return H.TF.normalize(t.expand(3, -1, -1).clone(), H.IMAGENET_MEAN, H.IMAGENET_STD).unsqueeze(0)


def explain(image_path, cam_fn, class_idx=1):
    """THE PRODUCTION CALL.

    Returns (prob_malignant, heatmap) where heatmap.shape == (orig_H, orig_W) -- the size of the
    image the caller supplied, NOT the 384x640 preprocessed size.
    """
    with Image.open(winlong(image_path)) as im:
        pil = im.convert("L")
        ow, oh = pil.size
        x = preprocess(pil).to(H.DEVICE)
    cam, probs = cam_fn(x, class_idx=class_idx)
    return float(probs[0, 1]), H.cam_to_original(cam[0], ow, oh)


print("explain() returns a heatmap at the ORIGINAL image resolution.")

### Round-trip check

A synthetic marker at a known fraction of the source must return to that same fraction after
`fit_pad` -> `cam_to_original`, across the real aspect ratios in the dataset.

In [ ]:
print(f"{'source (WxH)':>18} | {'padded core':>13} | {'returned (WxH)':>15} | rel. error (x, y)")
worst = 0.0
for (ow, oh) in [(2986, 5356), (3024, 5063), (4144, 6736), (2041, 4384), (3552, 5944)]:
    nw, nh, ox, oy = H.fit_pad_geometry(ow, oh)
    cam = np.zeros((H.HIRES_H, H.HIRES_W), np.float32)
    cy, cx = oy + int(nh * 0.35), ox + int(nw * 0.65)
    yy, xx = np.ogrid[:H.HIRES_H, :H.HIRES_W]
    cam[(yy - cy) ** 2 + (xx - cx) ** 2 <= 8 ** 2] = 1.0
    back = H.cam_to_original(cam, ow, oh)
    py, px = np.unravel_index(np.argmax(back), back.shape)
    ey, ex = abs(py / oh - 0.35), abs(px / ow - 0.65)
    worst = max(worst, ey, ex)
    print(f"{ow:8d}x{oh:<8d} | {nw:4d}x{nh:<7d} | {back.shape[1]:6d}x{back.shape[0]:<7d} | "
          f"({ex:.4f}, {ey:.4f})")
assert worst < 0.02, f"round-trip error {worst:.4f} too large"
print(f"\nworst relative error {worst:.4f} -- output size matches input size exactly")

## 2. Load a model

Today: the dual-input teacher's `full_extractor`. When hi-res single-input checkpoints exist, set
`MODE = "single"` -- nothing else changes, which is the point of hooking rather than editing the
model.

In [ ]:
MODE = "teacher"          # "teacher" | "single"
ARCH = "densenet121"
SINGLE_CKPT = None        # e.g. ".../stage0_hires_segaux_densenet121.pth"


class TeacherFullBranch(torch.nn.Module):
    """Exposes the teacher's full-image branch with a SingleInputModel-shaped interface so the
    identical Grad-CAM code runs on both. The crop branch is zero-filled: we are explaining what
    the full-image pathway responds to, and the deployed model has no crop at all."""

    def __init__(self, teacher):
        super().__init__()
        self.teacher = teacher
        # drop the trailing AdaptiveAvgPool2d so `extractor` yields a SPATIAL map
        self.extractor = torch.nn.Sequential(*[m for m in teacher.full_extractor
                                               if not isinstance(m, torch.nn.AdaptiveAvgPool2d)])

    def forward(self, x):
        spatial = self.extractor(x)
        f = torch.flatten(torch.nn.functional.adaptive_avg_pool2d(spatial, 1), 1)
        return self.teacher.classifier(torch.cat([f, torch.zeros_like(f)], dim=1))


def load_model():
    if MODE == "teacher":
        m = TeacherFullBranch(H.load_teacher(ARCH, CKPT_DIR)).to(H.DEVICE).eval()
        return m, f"teacher full-branch ({ARCH})"
    ck = torch.load(SINGLE_CKPT, map_location=H.DEVICE, weights_only=False)
    assert list(ck.get("input_size", [])) == [H.HIRES_H, H.HIRES_W], (
        f"checkpoint input_size {ck.get('input_size')} != {[H.HIRES_H, H.HIRES_W]}. A 224-trained "
        "state dict loads without error at 640x384 and is silently wrong.")
    m = H.SingleInputModel(ARCH, pooling=ck.get("pooling", "gap"),
                           use_seg=ck.get("use_seg", False)).to(H.DEVICE)
    m.load_state_dict(ck["model_state_dict"])
    m.eval()
    return m, f"single-input {ARCH} ({os.path.basename(SINGLE_CKPT)})"


model, MODEL_NOTE = load_model()
with torch.no_grad():
    probe = model.extractor(torch.zeros(1, 3, H.HIRES_H, H.HIRES_W, device=H.DEVICE))
print(f"{MODEL_NOTE}\nspatial map {tuple(probe.shape)} -> Grad-CAM grid "
      f"{probe.shape[-2]}x{probe.shape[-1]}, each cell ~{H.HIRES_H // probe.shape[-2]}px")
del probe
H.free_gpu()

**That grid size is the resolution of your explanation.** On the old 224 cache it was 7x7 with
the median lesion at 19x10 px -- smaller than one cell, so Grad-CAM could not have localized a
lesion even in principle. At 640x384 it is 20x12 with the lesion at 35x34 px: about one cell.
Coarse, but now genuinely capable of pointing.

## 3. Correctness checks before trusting any number

A localization metric that scores high for a random model is measuring the *dataset*, not the
model. These run first so the numbers that follow mean something.

In [ ]:
def load_mask(row):
    return np.array(Image.open(row[H.ROI_HIRES_COL]).convert("L")) > 127


def tensor_from_cached(row):
    t = H.TF.to_tensor(Image.open(row[H.FULL_HIRES_COL]).convert("L"))
    return H.TF.normalize(t.expand(3, -1, -1).clone(), H.IMAGENET_MEAN, H.IMAGENET_STD).unsqueeze(0)


def cam_for_row(row, cam_fn, class_idx=1):
    cam, probs = cam_fn(tensor_from_cached(row).to(H.DEVICE), class_idx=class_idx)
    return cam[0], float(probs[0, 1])


# --- (a) a random-init model must score at chance ---
sub = test_df.head(40)
rand_model = H.SingleInputModel(ARCH, pooling="gap", use_seg=False).to(H.DEVICE).eval()
rand_hits, rand_ratio, chance = [], [], []
with H.GradCAM(rand_model) as cf:
    for _, row in tqdm(sub.iterrows(), total=len(sub), desc="random-init", leave=False):
        cam, _ = cam_for_row(row, cf)
        mask, v = load_mask(row), H.validity_mask(*source_size(row))
        if not mask.any():
            continue
        m = H.localization_metrics(cam, mask, v)
        rand_hits.append(m["pointing_hit"])
        rand_ratio.append(m["concentration_ratio"])
        chance.append(m["chance_rate"])
del rand_model
H.free_gpu()

print(f"random-init model : pointing {np.mean(rand_hits) * 100:5.1f}%   "
      f"concentration {np.nanmean(rand_ratio):.2f}x")
print(f"chance            : lesion covers {np.mean(chance) * 100:.2f}% of valid pixels")
print("\nA random model scoring ~1x concentration means the metric measures the MODEL, not the")
print("geometry of the dataset. A trained model must score well above this.")

In [ ]:
# --- (b) spatial correspondence against a known answer ---
# NOT an hflip-equivariance test: convolutions are translation-equivariant but NOT reflection-
# equivariant, so CAM(flip(x)) != flip(CAM(x)) even for perfectly correct code. Instead a stride-32
# conv maps input block [32i:32i+32, 32j:32j+32] to cell (i,j) exactly, so a bright patch at a
# known place MUST produce a CAM peak there. This also pins down H/W ordering in the upsample.
class _StrideNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.extractor = torch.nn.Conv2d(3, 8, 32, stride=32, bias=False)
        torch.nn.init.constant_(self.extractor.weight, 0.01)
        self.classifier = torch.nn.Linear(8, 2)
        torch.nn.init.constant_(self.classifier.weight, 0.5)
        torch.nn.init.zeros_(self.classifier.bias)

    def forward(self, x):
        a = self.extractor(x)
        return self.classifier(torch.flatten(torch.nn.functional.adaptive_avg_pool2d(a, 1), 1))


snet = _StrideNet().to(H.DEVICE).eval()
for (ry, rx) in [(0.25, 0.75), (0.80, 0.20), (0.50, 0.50)]:
    xin = torch.zeros(1, 3, H.HIRES_H, H.HIRES_W, device=H.DEVICE)
    ty, tx = int(H.HIRES_H * ry), int(H.HIRES_W * rx)
    xin[0, :, ty:ty + 32, tx:tx + 32] = 5.0
    with H.GradCAM(snet) as cf:
        c, _ = cf(xin)
    py, px = np.unravel_index(np.argmax(c[0]), c[0].shape)
    print(f"  patch (y={ty:3d}, x={tx:3d}) -> CAM peak (y={py:3d}, x={px:3d})   "
          f"err ({abs(py - ty - 16)}, {abs(px - tx - 16)}) px")
    assert abs(py - ty - 16) <= 32 and abs(px - tx - 16) <= 32, "spatial mapping is wrong"
del snet
H.free_gpu()
print("\nCAM peaks track the driving input region in BOTH axes -- no transposition.")

## 4. Overlay: original image + ROI + Grad-CAM

All at **original resolution** -- what a radiologist would actually be shown. The green contour is
ground truth; the heatmap is the model's explanation. Whether they agree is answered numerically in
section 5.

In [ ]:
DISP_DIV = 6      # display downscale; a full-res float heatmap per panel would exhaust host RAM


def overlay_panel(row, cam_fn, ax_row, show_titles=False):
    src = source_path(row)
    with Image.open(winlong(src)) as im:
        pil = im.convert("L")
        ow, oh = pil.size
        dw, dh = ow // DISP_DIV, oh // DISP_DIV
        disp = np.asarray(pil.resize((dw, dh), Image.BILINEAR), np.float32) / 255.0

    prob, heat_full = explain(src, cam_fn)
    heat = np.asarray(Image.fromarray(heat_full, mode="F").resize((dw, dh), Image.BILINEAR),
                      np.float32)
    del heat_full                       # ~61MB at full res -- release immediately

    # the cached mask lives in PADDED space; strip the padding before display
    nw, nh, ox, oy = H.fit_pad_geometry(ow, oh)
    core = load_mask(row)[oy:oy + nh, ox:ox + nw].astype(np.float32)
    mask_d = np.asarray(Image.fromarray(core, mode="F").resize((dw, dh), Image.NEAREST)) > 0.5

    truth = "Malignant" if H.encode_label(row[H.LABEL_COL]) == 1 else "Benign"
    ax_row[0].imshow(disp, cmap="gray")
    ax_row[1].imshow(disp, cmap="gray")
    ax_row[1].contour(mask_d, [0.5], colors="lime", linewidths=1.4)
    ax_row[2].imshow(disp, cmap="gray")
    ax_row[2].imshow(heat, cmap="jet", alpha=0.45)
    ax_row[3].imshow(disp, cmap="gray")
    ax_row[3].imshow(heat, cmap="jet", alpha=0.40)
    ax_row[3].contour(mask_d, [0.5], colors="lime", linewidths=1.4)
    if show_titles:
        for a, t in zip(ax_row, ["original", "+ ROI (truth)", "+ Grad-CAM", "both"]):
            a.set_title(t, fontsize=10)
    ax_row[0].set_ylabel(f"{truth}\np(malig)={prob:.2f}", fontsize=8)
    for a in ax_row:
        a.set_xticks([])
        a.set_yticks([])


lab = test_df[H.LABEL_COL].apply(H.encode_label)
show = pd.concat([test_df[lab == 1].head(3), test_df[lab == 0].head(2)])
fig, axes = plt.subplots(len(show), 4, figsize=(13, 3.1 * len(show)))
with H.GradCAM(model) as cf:
    for i, (_, row) in enumerate(show.iterrows()):
        overlay_panel(row, cf, axes[i], show_titles=(i == 0))
plt.suptitle(f"Grad-CAM vs ground-truth ROI  -  {MODEL_NOTE}", y=1.002)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "gradcam_overlay_grid.png"), dpi=110, bbox_inches="tight")
plt.show()

## 5. Does the heatmap agree with the ROI? (measured)

| metric | what it says |
|---|---|
| **Pointing game** | does the single hottest pixel fall inside the lesion? the standard WSOL metric |
| **Pointing @15px** | same, forgiving a one-cell upsampling offset |
| **Energy concentration** | fraction of total heat landing on the lesion; threshold-free |
| **Concentration ratio** | the above / chance -- *"N times more attention on the lesion than random"*. The headline. |
| **Pixel AP / AUROC** | heat values as scores against the mask; threshold-free |
| **Peak-in-padding** | diagnostic: if high, the model keys on canvas geometry rather than tissue |

Computed in 384x640 space where the mask already lives -- mathematically equivalent to computing at
original resolution and vastly cheaper than materializing a 61 MB float array per image.

In [ ]:
def evaluate_localization(df, cam_fn, class_idx=1, limit=None, desc="localization"):
    rows = []
    it = df if limit is None else df.head(limit)
    for _, row in tqdm(it.iterrows(), total=len(it), desc=desc, leave=False):
        mask = load_mask(row)
        if not mask.any():
            continue
        cam, prob = cam_for_row(row, cam_fn, class_idx)
        v = H.validity_mask(*source_size(row))
        m = H.localization_metrics(cam, mask, v)
        for tau in (0.3, 0.5, 0.7):
            m[f"iou@{tau}"] = H.iou_at(cam, mask, v, tau)
        m.update(prob_malignant=prob, label=H.encode_label(row[H.LABEL_COL]))
        rows.append(m)
    return pd.DataFrame(rows)


with H.GradCAM(model) as cf:
    loc = evaluate_localization(test_df, cf, desc="test localization")
loc.to_csv(os.path.join(RESULTS_DIR, f"localization_{MODE}_{ARCH}.csv"), index=False)


def summarize(d, name):
    return {"model": name, "n": len(d),
            "pointing_%": 100 * d["pointing_hit"].mean(),
            "pointing@15px_%": 100 * d["pointing_hit_tol"].mean(),
            "energy_conc": d["energy_concentration"].mean(),
            "conc_ratio_x": d["concentration_ratio"].mean(),
            "pixel_ap": d["pixel_ap"].mean(),
            "pixel_auroc": d["pixel_auroc"].mean(),
            "iou@0.5": d["iou@0.5"].mean(),
            "peak_in_padding_%": 100 * d["peak_in_padding"].mean(),
            "median_peak_dist_px": d["peak_dist_px"].median()}


pd.set_option("display.width", 200)
print(pd.DataFrame([summarize(loc, MODEL_NOTE)]).T.to_string(header=False))

# bootstrap CI on the headline -- at n~319 a bare point estimate invites noise-chasing
rng = np.random.default_rng(H.SEED)
vals = loc["concentration_ratio"].dropna().values
boot = [np.mean(rng.choice(vals, len(vals), replace=True)) for _ in range(2000)]
print(f"\nconcentration ratio {vals.mean():.2f}x  95% CI "
      f"[{np.percentile(boot, 2.5):.2f}, {np.percentile(boot, 97.5):.2f}]")
print(f"chance would be 1.00x; mean lesion covers {loc['chance_rate'].mean() * 100:.2f}% "
      "of valid pixels")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

axes[0].hist(loc["concentration_ratio"].clip(0, 30), bins=40, color="#4C78A8")
axes[0].axvline(1.0, color="crimson", ls="--", label="chance (1.0x)")
axes[0].set_title("Concentration ratio")
axes[0].set_xlabel("x chance")
axes[0].legend()

taus = np.linspace(0.05, 0.95, 19)
curves = []
with H.GradCAM(model) as cf:
    for _, row in tqdm(test_df.head(60).iterrows(), total=60, desc="IoU curve", leave=False):
        mask = load_mask(row)
        if not mask.any():
            continue
        cam, _ = cam_for_row(row, cf)
        v = H.validity_mask(*source_size(row))
        curves.append([H.iou_at(cam, mask, v, t) for t in taus])
axes[1].plot(taus, np.nanmean(curves, axis=0), marker="o", ms=3)
axes[1].set_title("IoU vs threshold (n=60)")
axes[1].set_xlabel("tau")
axes[1].set_ylabel("IoU")

for lbl, nm, c in [(1, "Malignant", "#E45756"), (0, "Benign", "#54A24B")]:
    axes[2].hist(loc[loc["label"] == lbl]["concentration_ratio"].clip(0, 30),
                 bins=25, alpha=0.6, label=nm, color=c)
axes[2].set_title("Concentration by class")
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "localization_summary.png"), dpi=110)
plt.show()

print("Absolute IoU will look low. That is a property of the task, not a failure: the lesion covers")
print(f"~{loc['chance_rate'].mean() * 100:.2f}% of the image, so even a well-placed but coarse 20x12")
print("heatmap cannot reach high overlap. Report the curve and the concentration ratio, not a")
print("single IoU number.")

### Reading the numbers above

Compare `conc_ratio_x` against **1.00x (chance)** and against the random-init model from section 3.

Measured on 2026-08-07 with the teacher full-branch: random-init scored **1.28x**, the teacher
**1.55x [95% CI 1.40-1.71]**, pixel AUROC 0.634. The teacher sits only a little above random --
which is exactly what the two distribution shifts noted at the top of this notebook predict
(trained at 224, run at 640x384; crop branch zeroed). **These are not thesis numbers.**

What they *do* establish is that the pipeline is sound end to end: sizes round-trip exactly, CAM
peaks track a known input patch in both axes, a random model scores at chance, and a real model
scores above it. When a properly trained hi-res single-input model is dropped in, a genuine
localization signal should push this ratio substantially higher -- and if it does not, that itself
is the finding to report.

A useful reference point: the seg-aux model is trained with an explicit mask-prediction objective,
so if auxiliary supervision works at all, its concentration ratio should clearly exceed the
control's. Section 6 tests that difference for significance rather than eyeballing it.

## 6. Comparing two models

The thesis claim is that **auxiliary ROI-mask supervision produces better explanations**. That is a
paired comparison on identical images, so it needs paired tests: McNemar's exact for the binary
pointing game, Wilcoxon signed-rank for the continuous metrics (bounded and skewed, so normality
should not be assumed).

**Two fairness rules, or the comparison is worthless:**

1. The seg-aux model's CAM must come from the **classifier** path -- gradients of the class-1 logit
   w.r.t. `extractor` -- *not* from the segmentation decoder's output. Otherwise you are comparing a
   segmentation model's segmentation against a classifier's Grad-CAM, which is not the claim. The
   decoder output may be reported separately as a clearly labelled auxiliary-localization row.
2. Both models must share resolution, batch size, epoch budget, seed and pooling. Only the seg loss
   term differs.

Fill in the two checkpoints once the hi-res runs exist; the cell no-ops until then.

In [ ]:
COMPARE = {
    # "control": "<path to hi-res control checkpoint>",
    # "segaux":  "<path to hi-res segaux checkpoint>",
}

if len(COMPARE) < 2:
    print("Set COMPARE to two hi-res checkpoints after training to run the paired comparison.")
else:
    from scipy import stats
    results = {}
    for name, path in COMPARE.items():
        ck = torch.load(path, map_location=H.DEVICE, weights_only=False)
        assert list(ck.get("input_size", [])) == [H.HIRES_H, H.HIRES_W], f"{name}: wrong input_size"
        m = H.SingleInputModel(ARCH, pooling=ck.get("pooling", "gap"),
                               use_seg=ck.get("use_seg", False)).to(H.DEVICE)
        m.load_state_dict(ck["model_state_dict"])
        m.eval()
        with H.GradCAM(m) as cf:            # classifier path, NOT the seg decoder
            results[name] = evaluate_localization(test_df, cf, desc=name)
        del m
        H.free_gpu()

    a, b = list(COMPARE)
    da, db = results[a], results[b]
    print(pd.DataFrame([summarize(da, a), summarize(db, b)]).to_string(index=False))

    x, y = da["pointing_hit"].values.astype(bool), db["pointing_hit"].values.astype(bool)
    n01, n10 = int((~x & y).sum()), int((x & ~y).sum())
    p_mc = stats.binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if (n01 + n10) else 1.0
    print(f"\nMcNemar (pointing game): {b} wins {n01}, {a} wins {n10}, p = {p_mc:.4f}")

    for metric in ("energy_concentration", "concentration_ratio", "pixel_ap"):
        s, p = stats.wilcoxon(da[metric], db[metric], nan_policy="omit")
        print(f"Wilcoxon {metric:22s}: {a} {da[metric].mean():.4f} vs "
              f"{b} {db[metric].mean():.4f}   p = {p:.4f}")

## 7. What to record for the thesis

- **Concentration ratio is the headline**: *"the heatmap places N times more attention on the lesion
  than chance"* -- threshold-free, scale-invariant, immediately interpretable.
- **Report the IoU curve, not a single IoU.** With a target under 1% of the image, absolute IoU is
  low by construction and a single number invites a misreading.
- **Bootstrap 95% CIs on every headline figure.** At n=319 the SE on a proportion is ~2.6 pp, so a
  bare point estimate is not defensible.
- **Report the peak-in-padding rate even when it is near zero** -- it demonstrates the model is not
  exploiting the canvas.
- If seg-aux improves localization but not accuracy, **report exactly that**. Explainability gained
  at no accuracy cost is a real, publishable result; do not reshape it into an accuracy story.